# 6 - Baseline departure-delay model

## 1 - Load data and define the target

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_csv(
    PROJECT_ROOT / "data/processed/flights_weather_2024_top10.csv.gz",
    dtype={"CRSDepTime": "string"},
    parse_dates=["FlightDate"],
)

assert df["DepDel15"].isin([0, 1]).all()

display(
    df.groupby("Origin").agg(
        flights=("DepDel15", "size"),
        delay_rate=("DepDel15", "mean"),
    ).round(3)
)

,flights,delay_rate
Origin,,
ATL,338500,0.197
CLT,214578,0.264
DEN,305160,0.232
DFW,307745,0.275
LAS,187579,0.235
LAX,192471,0.180
LGA,158586,0.180
ORD,276267,0.222
PHX,192373,0.201


## Define development split

In [6]:
TEST_AIRPORT = "ATL"
VAL_AIRPORT = "LGA"

train = df.loc[~df["Origin"].isin([TEST_AIRPORT, VAL_AIRPORT]) & df["FlightDate"].lt("2024-10-01")].copy()
val = df.loc[df["Origin"].eq(VAL_AIRPORT) & df["FlightDate"].ge("2024-10-01")].copy()

assert len(train) > 0 and len(val) > 0
assert set(train["Origin"]).isdisjoint(val["Origin"])
assert TEST_AIRPORT not in set(train["Origin"]) | set(val["Origin"])
assert train["FlightDate"].max() < val["FlightDate"].min()

display(pd.DataFrame({
    "flights": [len(train), len(val)],
    "delay_rate": [train["DepDel15"].mean(), val["DepDel15"].mean()],
}, index=["train", "val"]))

,flights,delay_rate
train,1370927,0.248886
val,40373,0.147921


## Prepare features + train baseline

In [7]:
%pip install scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
for data in [train, val]:
    data["departure_hour"] = data["CRSDepTime"].astype(int) // 100

categorical = ["Reporting_Airline", "Dest", "Month", "DayOfWeek", "departure_hour"]
numeric = ["Distance"]
features = categorical + numeric

X_train, y_train = train[features], train["DepDel15"].astype(int)
X_val, y_val = val[features], val["DepDel15"].astype(int)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", StandardScaler(), numeric),
])

baseline = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))

baseline.fit(X_train, y_train)

val_probability = baseline.predict_proba(X_val)[:, 1]
print("Baseline fitted, validation probabilities generated")

Baseline fitted, validation probabilities generated


## Evaluate baseline

In [10]:
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss,
)

predictions = {
    "Constant baseline": np.full(len(y_val), y_train.mean()),
    "Logistic regression": val_probability,
}

results = pd.DataFrame({
    name: {
        "average_precision": average_precision_score(y_val, probability),
        "roc_auc": roc_auc_score(y_val, probability),
        "log_loss": log_loss(y_val, probability),
    }
    for name, probability in predictions.items()
}).T

display(results.round(4))
print(f"Validation delay rate: {y_val.mean():.2%}")

,average_precision,roc_auc,log_loss
Constant baseline,0.1479,0.5000,0.4496
Logistic regression,0.2435,0.6508,0.4155


Validation delay rate: 14.79%


## Add weather features

In [11]:
from sklearn.impute import SimpleImputer

weather_features = ["temperature", "dew_point_temperature", "wind_speed", "visibility"]
weather_numeric = numeric + weather_features
weather_cols = categorical + weather_numeric

weather_preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), weather_numeric),
])

weather_model = make_pipeline(weather_preprocessor, LogisticRegression(max_iter=1000))

weather_model.fit(train[weather_cols], y_train)
weather_probability = weather_model.predict_proba(val[weather_cols])[:, 1]

results.loc["Logistic regression + weather"] = [
    average_precision_score(y_val, weather_probability),
    roc_auc_score(y_val, weather_probability),
    log_loss(y_val, weather_probability),
]

display(results.round(4))

,average_precision,roc_auc,log_loss
Constant baseline,0.1479,0.5000,0.4496
Logistic regression,0.2435,0.6508,0.4155
Logistic regression + weather,0.2751,0.6768,0.4145


## Save results of baseline model

In [12]:
OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_pilot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results.assign(
    test_airport=TEST_AIRPORT,
    val_airport=VAL_AIRPORT,
    training_period="2024-01 to 2024-09",
    val_period="2024-10 to 2024-12",
    training_flights=len(train),
    val_flights=len(val),
    val_delay_rate=y_val.mean(),
).to_csv(OUTPUT_DIR / "logistic_regression_results.csv", index_label="model")

## Validate across development airports

In [13]:
from sklearn.base import clone

df["departure_hour"] = df["CRSDepTime"].astype(int) // 100
development = df.loc[df["Origin"].ne(TEST_AIRPORT)]
rows = []

for airport in sorted(development["Origin"].unique()):
    source = development.loc[development["Origin"].ne(airport) & development["FlightDate"].lt("2024-10-01")]
    target = development.loc[development["Origin"].eq(airport) & development["FlightDate"].ge("2024-10-01")]

    y_source = source["DepDel15"].astype(int)
    y_target = target["DepDel15"].astype(int)

    probabilities = {"Constant baseline": np.full(len(target), y_source.mean())}

    for name, template, cols in [("Flight only", baseline, features), ("With weather", weather_model, weather_cols)]:
        model = clone(template)
        model.fit(source[cols], y_source)
        probabilities[name] = model.predict_proba(target[cols])[:, 1]

    for name, probability in probabilities.items():
        rows.append({
            "airport": airport,
            "model": name,
            "flights": len(target),
            "delay_rate": y_target.mean(),
            "average_precision": average_precision_score(y_target, probability),
            "roc_auc": roc_auc_score(y_target, probability),
            "log_loss": log_loss(y_target, probability),
        })

    print(f"{airport}: completed")

airport_results = pd.DataFrame(rows)
airport_results.to_csv(OUTPUT_DIR / "logistic_regression_airport_val.csv", index=False)

CLT: completed
DEN: completed
DFW: completed
LAS: completed
LAX: completed
LGA: completed
ORD: completed
PHX: completed
SEA: completed


display average across airports

In [14]:
metrics = ["average_precision", "roc_auc", "log_loss"]

display(airport_results.groupby("model")[metrics].mean().round(4))

ap_by_airport = airport_results.pivot(index="airport", columns="model", values="average_precision")
ap_by_airport["weather_gain"] = (ap_by_airport["With weather"] - ap_by_airport["Flight only"])

display(ap_by_airport.round(4))

,average_precision,roc_auc,log_loss
model,,,
Constant baseline,0.1718,0.5000,0.4746
Flight only,0.2424,0.6196,0.4687
With weather,0.2584,0.6320,0.4678


model,Constant baseline,Flight only,With weather,weather_gain
airport,,,,
CLT,0.1903,0.2565,0.2672,0.0107
DEN,0.1694,0.2450,0.2711,0.0260
DFW,0.2160,0.2952,0.3232,0.0280
LAS,0.1778,0.2629,0.2711,0.0082
LAX,0.1480,0.1967,0.2013,0.0047
LGA,0.1479,0.2435,0.2751,0.0316
ORD,0.1685,0.2294,0.2496,0.0202
PHX,0.1690,0.2353,0.2386,0.0033
SEA,0.1597,0.2175,0.2284,0.0110


## Random forest pilot

In [15]:
from sklearn.ensemble import RandomForestClassifier

forest_settings = {
    "n_estimators": 200,
    "max_depth": 16,
    "min_samples_leaf": 50,
    "max_samples": 0.3,
    "n_jobs": -1,
    "random_state": 42,
}

forest_models = {}

for name, preparation, columns in [
    ("Random forest", preprocessor, features),
    ("Random forest + weather", weather_preprocessor, weather_cols),
]:
    model = make_pipeline(
        clone(preparation),
        RandomForestClassifier(**forest_settings),
    )

    model.fit(train[columns], y_train)
    probability = model.predict_proba(val[columns])[:, 1]
    forest_models[name] = model

    results.loc[name, metrics] = [
        average_precision_score(y_val, probability),
        roc_auc_score(y_val, probability),
        log_loss(y_val, probability),
    ]

    print(f"{name}: completed")

display(results.round(4))
results.to_csv(OUTPUT_DIR / "pilot_model_comparison.csv", index_label="model")

Random forest: completed
Random forest + weather: completed


,average_precision,roc_auc,log_loss
Constant baseline,0.1479,0.5000,0.4496
Logistic regression,0.2435,0.6508,0.4155
Logistic regression + weather,0.2751,0.6768,0.4145
Random forest,0.2318,0.6384,0.4273
Random forest + weather,0.2578,0.6603,0.4213


## Validate RF performance across development airports

In [16]:
forest_rows = []

for airport in sorted(development["Origin"].unique()):
    source = development.loc[development["Origin"].ne(airport) & development["FlightDate"].lt("2024-10-01")]
    target = development.loc[development["Origin"].eq(airport) & development["FlightDate"].ge("2024-10-01")]

    y_source = source["DepDel15"].astype(int)
    y_target = target["DepDel15"].astype(int)

    for name, cols in [("Random forest", features), ("Random forest + weather", weather_cols)]:
        model = clone(forest_models[name])
        model.fit(source[columns], y_source)
        probability = model.predict_proba(target[columns])[:, 1]

        forest_rows.append({
            "airport": airport,
            "model": name,
            "flights": len(target),
            "delay_rate": y_target.mean(),
            "average_precision": average_precision_score(y_target, probability),
            "roc_auc": roc_auc_score(y_target, probability),
            "log_loss": log_loss(y_target, probability),
        })

    print(f"{airport}: completed")

forest_results = pd.DataFrame(forest_rows)
forest_results.to_csv(OUTPUT_DIR / "random_forest_airport_validation.csv", index=False)

CLT: completed
DEN: completed
DFW: completed
LAS: completed
LAX: completed
LGA: completed
ORD: completed
PHX: completed
SEA: completed


In [17]:
comparison = pd.concat([
    airport_results.replace({"model": {
        "Flight only": "Logistic regression",
        "With weather": "Logistic regression + weather",
    }}),
    forest_results,
], ignore_index=True)

display(comparison.groupby("model")[metrics].mean().round(4))
display(comparison.pivot(index="airport", columns="model", values="average_precision").round(4))

comparison.to_csv(OUTPUT_DIR / "model_comparison_airport_validation.csv", index=False)

,average_precision,roc_auc,log_loss
model,,,
Constant baseline,0.1718,0.5000,0.4746
Logistic regression,0.2424,0.6196,0.4687
Logistic regression + weather,0.2584,0.6320,0.4678
Random forest,0.2375,0.6134,0.4621
Random forest + weather,0.2471,0.6234,0.4571


model,Constant baseline,Logistic regression,Logistic regression + weather,Random forest,Random forest + weather
airport,,,,,
CLT,0.1903,0.2565,0.2672,0.2552,0.2619
DEN,0.1694,0.2450,0.2711,0.2416,0.2570
DFW,0.2160,0.2952,0.3232,0.2918,0.2983
LAS,0.1778,0.2629,0.2711,0.2588,0.2627
LAX,0.1480,0.1967,0.2013,0.1933,0.1935
LGA,0.1479,0.2435,0.2751,0.2318,0.2578
ORD,0.1685,0.2294,0.2496,0.2212,0.2337
PHX,0.1690,0.2353,0.2386,0.2233,0.2277
SEA,0.1597,0.2175,0.2284,0.2208,0.2312


## Findings

weather improved airport-averaged precision, ROC-AUC, and log loss for both models. Logistic regression combined with weather achieved the highest average precision (0.2584) and ROC-AUC (0.6320). Random forest with weather achieved the lowest log loss (0.4571).

## Model-selection criterion

The primary selection metric is mean average precision across validation airports, with equal weight per airport. ROC-AUC and log loss are secondary. This choice was made after exploratory results from logistic regression and random forest models, before evaluating 2025 test data

In [18]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.9 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Gradient boosting pilot

In [20]:
from xgboost import XGBClassifier

boost_settings = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": 4,
    "random_state": 42,
}

boost_models = {}

for name, preparation, cols in [
    ("XGBoost", preprocessor, features),
    ("XGBoost + weather", weather_preprocessor, weather_cols),
]:
    model = make_pipeline(
        clone(preparation),
        XGBClassifier(**boost_settings),
    )
    model.fit(train[columns], y_train)
    probability = model.predict_proba(val[columns])[:, 1]
    boost_models[name] = model

    results.loc[name, metrics] = [
        average_precision_score(y_val, probability),
        roc_auc_score(y_val, probability),
        log_loss(y_val, probability),
    ]

    print(f"{name}: completed")

display(results.round(4))
results.to_csv(OUTPUT_DIR / "pilot_model_comparison.csv", index_label="model")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/alainbouscholte/Developer/Thesis/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <09B43001-8527-34F6-AAE4-B97DA8A6CBC0> /Users/alainbouscholte/Developer/Thesis/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebr